In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("weather_dev.silver.weather_forecast_hourly")

df_gold = (
    df_silver
    .groupBy("forecast_date", "latitude", "longitude")
    .agg(
        F.round(F.avg("temperature_c"), 1).alias("avg_temperature_c"),
        F.round(F.max("temperature_c"), 1).alias("max_temperature_c"),
        F.round(F.min("temperature_c"), 1).alias("min_temperature_c"),
        F.round(F.sum("precipitation_mm"), 1).alias("total_precipitation_mm"),
        F.round(F.avg("wind_speed_kmh"), 1).alias("avg_wind_speed_kmh"),
        F.round(F.max("wind_speed_kmh"), 1).alias("max_wind_speed_kmh"),
    )
    .orderBy("forecast_date")
)

display(df_gold)

In [0]:
table_name = "weather_dev.gold.weather_daily_summary"

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("forecast_date")
    .saveAsTable(table_name)
)

In [0]:
display(spark.sql("SELECT * FROM weather_dev.gold.weather_daily_summary"))

In [0]:
#spark.sql("DELETE FROM weather_dev.gold.weather_daily_summary")

In [0]:
%sql
COMMENT ON TABLE weather_dev.gold.weather_daily_summary IS 'Resumen diario de clima para CDMX, agregado desde silver.weather_forecast_hourly';

ALTER TABLE weather_dev.gold.weather_daily_summary ALTER COLUMN forecast_date COMMENT 'Fecha del pronóstico (no la fecha de ingesta)';
ALTER TABLE weather_dev.gold.weather_daily_summary ALTER COLUMN avg_temperature_c COMMENT 'Temperatura promedio del día en grados Celsius';
ALTER TABLE weather_dev.gold.weather_daily_summary ALTER COLUMN total_precipitation_mm COMMENT 'Precipitación acumulada del día en milímetros';